<a href="https://colab.research.google.com/github/Lookieman/MSAI/blob/main/AI6130/AI6130_Grp/Iter1_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install langchain langchain-community sentence-transformers faiss-gpu torch

In [ ]:
import os
import logging
import torch
import faiss
import numpy as np
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional

# For document loading and processing
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# For embeddings
from sentence_transformers import SentenceTransformer

In [ ]:
def get_papers_dir() -> str:

  try:
    from google.colab import drive
    drive.mount('/content/drive')
    papers_dir = '/content/drive/MyDrive/AI6130_Grp'

  except ImportError:
    papers_dir = "C:/Users/luqma/AI6130/AI6130_Grp/papers"

  return papers_dir

In [ ]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("rag_system.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [ ]:
class RAGSystem:

  def __init__(self, embedding_model_name: str = "BAAI/bge-small-en-v1.5", chunz_size: int = 1500, chunk_overlap: int = 300):

    #init param
    self.embedding_model_name = embedding_model_name
    self.chunk_size = chunk_size
    self.chunk_overlap = chunk_overlap

    #init storage for document chunks and metadata

    self.documents = []
    self.meatadatas = []
    self.embeddings = None
    self.embedding_model = None
    self.index = None

    #load embedding model
    self._load_embedding_model()

    logger.info(f"Initialized RAG System with {embedding_model_name}")
    logger.info(f" Chunk size: {chunk_size}, Overlap is {chunk_overlap}")

  def _load_embedding_model(self):
    #Load embedding model

    try:
      logger.info(f"Loading embedding model: {self.embedding_model_name}")
      self.embedding_model = SentenceTransformer(self.embedding_model_name, device=device)
      logger.info(f"Embedding model laoded!")
      except Exception as e:
        logger.error{f"Failed to load embedding model: {str(e)}"}
        raise



In [ ]:
  def load_documents(self, papers_dir: str)-> Dict[str, str]:

    papers_dir = Path(papers_dir)
    if not papers_dir.exist():
      logger error(f"Directory not found: {papers_dir}")
      return {}

    paper_files = [ f for f in os.listdir(papers_dir) if f.endswith('.pdf')]

    if not paper_files:
      logger.error(f"No PDF files found in {papers_dir}")
      return{}


    paper_contents = {}

    for paper_file in paper_files:
      paper_path = os.path.join(papers_dir, paper_file)
      logger.info(f"Loading document: {paper_path}")

      try:

        #Use PyPDFLoader to load pdf doc
        loader = PyPDFLoader(paper_path)
        doc_sections = loader.load()

        if doc_sections:
          content = "\n\n".join([section.page_content for section in doc_sections]) #double newlines to preserve para.
          paper_contents([paper_file]) = content
          logger.info(f"Successfully loaded {paper_file} with ({len(content)} characters)")
        else:
          logger.warning(f"No content loaded from {paaper_file}")

    logger.info(f"Loaded {len(paper_contents)} documents")
    return paper_contents



In [ ]:
  def process_documents(seld, paper_contents: Dict[str, str]):
    logger.info(f"Start Processing Doc....")

    #init text splitter
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap, separators=["\n\n", "\n","","."," "])

    self.documents = []
    self.metadata  = []

    #Process each paper

    for paper_name, content in paper_contents.items():
      logger.info(f"Splitting document: {paper_name}")

      chunks = text_splitter.split_text(content)

      #Store chunks w metadata

      for i, chunk in enumerate(chunks):
        self.documents.append(chunk)
        self.metadatas.append({
            "source": paper_name,
            "chunk_id": i,
            "total_chunks": len(chunks)
        })

      logger.info(f"Created {len(chunks)} chunks from {paper_name}")


    logger.info(f"Processed {len(paper_contents)} documents into {len(self.documents)} chunks")

In [ ]:
def create_embedding(self):
  if not self.documents:
    logger.erro("No documents to create embeddings for")
    return

  logger.info(f"Creating embeddings for {len(self.documents)} chunks...")

  try:
    #generate embeddings for chunks
    #This convert text chunks into numerical vectors
    self.embeddings = self.embeding_model.encode(self.documents, show_progress_bar=True)

    #Convert to numpy for FAISS

    self.embeddings = np.array(self.embeddings).astype('float32')

    logger.info(f"Created embeddings with shape: {elf.embeddings.shape}")
  except Exception as e:
    logger.error(f"Error creating embeddings: {str(e)}")
    raise


def build_faiss_index(self, use_gpu=False):

  if self.embeddings is None or len(self.embeddings) == 0:
    logger.error("No embeddings to build index with")
    return

  try:
    #Get embedding dimension
    dimension = self.embeddings.shapre[1]

    if use_gpu and not torch.cuda.is_available():
      logger.warning("GPU requested but not available. Falling back to CPU.")
      use_gpu = False

    if use_gpu:
      logger.info(f"Building GPU-accelerated FAISS index with dim {dimension}")
      res= faiss.StandardGPUResources()  #GPU resources

      cpu_index = faiss.IndexFlatL2(dimension)
      self.index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
      logger.info("Successfully created GPU FAISS index")
    else:
      #CPU version
      logger.info(f"Building FAISS index with dimension {dimension}")
      self.index = faiss.IndexFlatL2(dimension)

    self.index.add(self.embeddings)

    logger.info(f"Built FAISS index with {self.index.ntotal} vectors")
  except Exception as e:
    logger.error(f"Error building FAISS index: {str(e)}")
    raise

In [ ]:
def query(self, question: str , top_k: int = 3) -> List[Dict[str, Any]]:

  if self.index is None:
    logger.error("No index available for querying")
    return[]

  try:

    #create embedding for question
    question_embedding = self.embedding_model.ecode([question])
    question_embedding = np.array(question_embedding).astype('float32')

    #search the index
    distances, indices = self.index.search(question_embedding, top_k)

    #get relevant doc & metadata
    results = []
    for i, idx in enumerate(indices[0]):
      if idx < len(self.documents):
        results.append({
            "content": self.documents[idx];
            "metadata": self.metadata[idx];
            "score": float(distances[0][1])
        })
    logger.info(f"Query: '{question}' returned {len(results)}")
    return results
  except Exception as e:
    logger.error{f"Error querying index: {str(e)}"}
  return[]



In [ ]:
def ingest_and_index(self, papers_dir: str, use_gpu: bool=False)-> bool:

  try:

    #Step 1: Load Docs
    paper_contents = self.load_documents(papers_dir)
    if not paper_contents:
      logger.error(f" No documents loaded. Aborting")
      return False

    #Step 2: Process documents into chunks
    self.process_documents(paper_contents)

    #Step 3: Create embeddings
    self.create_embeddings()

    #Step 4: Build FAISS index
    self.build_faiss_index(use_gpu = use_gpu)

    logger.info("Successfully completed ingestion and indexing")
    return True
  except exception as e:
    logger.error(f" Error in ingest and index: {str(e)}")
    return False

In [ ]:
def run_rag_example():

  papers_dir = get_papers_dir()
  use_gpu = torch.cuda.is_available()


  #Initialize RAG
  rag_system = RAGSystem(
      embedding_model_name = "BAAI/bge-small-en-v1.5",
      chunk_size = 1500,
      chunk_overlap = 300
  )

    #Ingest and index doc
  success = rag_system.ingest_and_index(papers_dir,use_gpu=use_gpu)

  if success:

    example_questions = [
       ""
       ""
       ""
       ""
    ]

    #Run queries and display results:
    for question in example_questions:
      print("\n" + "="*80)
      print(f"Question: {question}")
      print("="*80)

      results = rag_system.query(question, top_k=2)

      for i, result in enumerate(results):
        print(f"\nRESULT {i+1} (SCore: {result['score']:.4f})")
        print(f"Source: {result['metadata']['source']}, " + f"Chunk: {result['meatdata']['chunk_id']+1}/{result['meatdata']['total_chunks']}" )
        print("-"*80)
        print(result['content'][:800] + "...." if len(result['content']) > 800 else result['content'])
        print("-"*80)

  return


In [ ]:
# if __name__ == "__main__":
#     # Check for GPU availability
#     gpu_available = torch.cuda.is_available()
#     if gpu_available:
#         print(f"GPU detected: {torch.cuda.get_device_name(0)}")
#         use_gpu = input("Use GPU acceleration for FAISS? (y/n): ").lower() == 'y'
#     else:
#         print("No GPU detected, using CPU only")
#         use_gpu = False

#     # Run the example
#     papers_dir = input(f"Enter path to papers directory (default: ./papers): ") or "./papers"
#     run_rag_example(papers_dir=papers_dir, use_gpu=use_gpu)

#     # Interactive mode
#     interactive = input("\nEnter interactive query mode? (y/n): ").lower() == 'y'
#     if interactive:
#         # Re-use the RAG system from the example or create a new one
#         rag_system = RAGSystem(
#             embedding_model_name="BAAI/bge-small-en-v1.5",
#             chunk_size=1500,
#             chunk_overlap=300
#         )

#         # Ingest and index documents
#         rag_system.ingest_and_index(papers_dir, use_gpu=use_gpu)

#         # Interactive query loop
#         print("\nInteractive Query Mode - Type 'quit' to exit")
#         while True:
#             query = input("\nEnter your question: ")
#             if query.lower() in ['quit', 'exit', 'q']:
#                 break

#             results = rag_system.query(query, top_k=3)

#             if not results:
#                 print("No relevant results found.")
#                 continue

#             for i, result in enumerate(results):
#                 print(f"\nRESULT {i+1} (Score: {result['score']:.4f})")
#                 print(f"Source: {result['metadata']['source']}")
#                 print("-"*80)
#                 print(result['content'][:800] + "..." if len(result['content']) > 800 else result['content'])
#                 print("-"*80)

In [ ]:
if __name__ == "__main__":
    # Run the example
    run_rag_example()